# 1. **Initialise**

`CARS_DATA` database was created when initially uploading the datasets using 'Upload local files' interface.

In [1]:
%%sql -r dataframe_1
USE ROLE accountadmin;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE CARS_DATA;
USE SCHEMA PUBLIC;

UsageError: Cell magic `%%sql` not found.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import when, col, lit, iff, regexp_replace, trim, round, when
from snowflake.snowpark.types import IntegerType, FloatType

from scipy.stats import zscore

In [ ]:
# Connect to the live database to retrieve data.
session = get_active_session()

# 2. **Load the Data**

The datasets are not all standardised (`bclass` & `fiesta` do not conform with the others), therefore they cannot be merged into the same table as of right now.
- **Option 1**: Load each dataset individually through the 'Upload local files' interface.
- **Option 2**: Programmatically upload the data through SQL.

**Option 2** was chosen for learning and documentation.

In [ ]:
%%sql -r dataframe_3
-- A STAGE is where raw data is stored before being imported into the back-end database as tables.
CREATE OR REPLACE STAGE CARS_DATA_STAGE;

`CARS_DATA_STAGE` will be in the `PUBLIC` schema. <br/>
Upload the datasets into `CARS_DATA_STAGE` excluding `README.md`.

**Steps Taken**:
- Ingestion.
- Add data.
- Load files into a stage.
- Upload the 13 files.
- Select the stage path.

In [ ]:
%%sql -r dataframe_2
LIST @CARS_DATA_STAGE;

Creating a template csv format

In [ ]:
%%sql -r dataframe_4
-- A file format rule for each dataset within the stage.
CREATE OR REPLACE FILE FORMAT CARS_CSV_FORMAT
    TYPE = 'CSV'                                    -- Applies to csv type files.
    FIELD_DELIMITER = ','                           -- Seperates columns by comma.
    PARSE_HEADER = TRUE                             -- Use header name as columns for INFER SCHEMA.
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'              -- Account for text fields that contain commas.
    NULL_IF = ('NULL', 'NA', 'N/A', 'NaN', 'null', 'na', 'n/a', 'nan', 'UNKNOWN', 'Unknown', 'unknown', ''); -- Returns NULL for each value that matches the following.

In [ ]:
%%sql -r dataframe_6
-- Read the CSV files following the file so we don't have to fill out column descriptions for each dataset such as name and type.
SELECT * FROM TABLE (
    INFER_SCHEMA(
        LOCATION => '@CARS_DATA_STAGE',
        FILE_FORMAT => 'CARS_CSV_FORMAT'
    )
);

Excluding hyundi which uses `tax(£)` instead of `tax`.

There are three different table structures:
- Type A [9]: audi, bmw, ford, hyundi, mercedes, skoda, toyota, vauxhall, volkswagen.
- Type B [2]: bclass, fiesta.
- Type C [2]: cclass, focus.

Ideally I'd define a for loop to create a table for each dataset, but I don't know the technical implementation using SQL/Python. <br/>
So instead, each table is created individually using SQL, each with a different schema based on their structure type. <br/>
Doing so allows for a clear hierarchy and database organisation.

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE SCHEMA CARS_DATA.TYPE_A;
CREATE OR REPLACE SCHEMA CARS_DATA.TYPE_B;
CREATE OR REPLACE SCHEMA CARS_DATA.TYPE_C;
CREATE OR REPLACE SCHEMA CARS_DATA.MASTER;  -- Where the final unified view will be stored.

In [ ]:
%%sql -r dataframe_5
-- Individually creating tables for each dataset.

USE SCHEMA PUBLIC;

-- Type A: 9 fields.
CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.AUDI
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/audi.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.BMW
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/bmw.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.FORD
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/ford.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.HYUNDI
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/hyundi.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.MERCEDES
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/mercedes.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.SKODA
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/skoda.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );

CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.TOYOTA
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/toyota.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.VAUXHALL
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/vauxhall.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );


CREATE OR REPLACE TABLE CARS_DATA.TYPE_A.VOLKSWAGEN
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/volkswagen.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );





-- Type B: 11 fields.
-- BCLASS explicitly defined as all VARCHAR to avoid INFER_SCHEMA mistyping "engine size2" as NUMBER.
-- Fix by CoCo.
CREATE OR REPLACE TABLE CARS_DATA.TYPE_B.BCLASS (
    "model" VARCHAR,
    "year" VARCHAR,
    "price" VARCHAR,
    "transmission" VARCHAR,
    "mileage" VARCHAR,
    "fuel type" VARCHAR,
    "engine size" VARCHAR,
    "mileage2" VARCHAR,
    "fuel type2" VARCHAR,
    "engine size2" VARCHAR,
    "reference" VARCHAR
);


CREATE OR REPLACE TABLE CARS_DATA.TYPE_B.FIESTA
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/fiesta.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );





-- Type C: 7 fields.
CREATE OR REPLACE TABLE CARS_DATA.TYPE_C.CCLASS
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/cclass.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );

CREATE OR REPLACE TABLE CARS_DATA.TYPE_C.FOCUS
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM TABLE(INFER_SCHEMA(LOCATION => '@CARS_DATA_STAGE/focus.csv', FILE_FORMAT => 'CARS_CSV_FORMAT'))
    );

Shell tables were created, which need to be populated with the data.

In [ ]:
%%sql -r dataframe_12
USE SCHEMA PUBLIC;

-- Type A
COPY INTO CARS_DATA.TYPE_A.AUDI
    FROM @CARS_DATA_STAGE/audi.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.BMW
    FROM @CARS_DATA_STAGE/bmw.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

    
COPY INTO CARS_DATA.TYPE_A.FORD
    FROM @CARS_DATA_STAGE/ford.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.HYUNDI
    FROM @CARS_DATA_STAGE/hyundi.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.MERCEDES
    FROM @CARS_DATA_STAGE/mercedes.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.SKODA
    FROM @CARS_DATA_STAGE/skoda.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.TOYOTA
    FROM @CARS_DATA_STAGE/toyota.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.VAUXHALL
    FROM @CARS_DATA_STAGE/vauxhall.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_A.VOLKSWAGEN
    FROM @CARS_DATA_STAGE/volkswagen.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
    




-- Type B
COPY INTO CARS_DATA.TYPE_B.BClASS
    FROM @CARS_DATA_STAGE/bclass.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_B.FIESTA
    FROM @CARS_DATA_STAGE/fiesta.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;





-- Type C
COPY INTO CARS_DATA.TYPE_C.CClASS
    FROM @CARS_DATA_STAGE/cclass.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;


COPY INTO CARS_DATA.TYPE_C.FOCUS
    FROM @CARS_DATA_STAGE/focus.csv
    FILE_FORMAT = (FORMAT_NAME = 'CARS_CSV_FORMAT')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

# 3. **Version Control**

Master copy of the database as a safety net.

In [ ]:
-- Create a master copy of the working database containing the current contents and structure for rollback.
CREATE OR REPLACE DATABASE CARS_DATA_BACKUP CLONE CARS_DATA;
USE DATABASE CARS_DATA;

Use this prompt to rollback within a specified period.

In [ ]:
%%sql -r dataframe_10
-- Time variables => (-60 * 1 * 1) = 1 minutes.
SET time_seconds = -60;
SET time_minutes = 1;
SET time_hours = 1;

-- Path variables => CARS_DATA.TYPE_A.AUDI;
SET path_database = 'CARS_DATA';
SET path_schema = 'TYPE_A'
SET path_table = 'AUDI'

-- Rollback at a given period for a given path.
SELECT * FROM IDENTIFIER(CONCAT($path_database, '.', $path_schema, '.', $path_table)) AT(OFFSET => $time_seconds * $time_minutes * $time_hours);

# 4. **Data View**

In [ ]:
%%sql -r dataframe_11
SELECT * FROM CARS_DATA.TYPE_A.AUDI;

Due to the complexity of different table structures, tables of the same type will be unified first.

In [ ]:
%%sql -r dataframe_7
-- Type A
USE SCHEMA TYPE_A;

CREATE OR REPLACE VIEW TYPE_A_VIEW AS
SELECT * FROM AUDI
UNION ALL
SELECT * FROM BMW
UNION ALL
SELECT * FROM FORD
UNION ALL
SELECT * FROM HYUNDI
UNION ALL
SELECT * FROM MERCEDES
UNION ALL
SELECT * FROM SKODA
UNION ALL
SELECT * FROM TOYOTA
UNION ALL
SELECT * FROM VAUXHALL
UNION ALL
SELECT * FROM VOLKSWAGEN;





-- Type B
USE SCHEMA TYPE_B;

CREATE OR REPLACE VIEW TYPE_B_VIEW AS
SELECT * FROM BCLASS
UNION ALL
SELECT * FROM FIESTA;





-- Type C
USE SCHEMA TYPE_C;

CREATE OR REPLACE VIEW TYPE_C_VIEW AS
SELECT * FROM CCLASS
UNION ALL
SELECT * FROM FOCUS;

Unifying the three types into a single unified view. <br/>

Things to keep in mind:
- Due to data type misalignment between views, each field will be casted as a `VARCHAR` to avoid compatability issues with numbers and text.
    - Data typing will be revisited when cleaning the data.
- Columns exist within some datasets but not others.
    - `NULL` will be used to represent the absense of a column.
- Potential duplicate columns may exist from `TYPE_B_VIEW`.
    - `fuelType2`
    - `engineSize2`
    - `mileage2`

In [ ]:
%%sql -r dataframe_14
USE SCHEMA MASTER;

CREATE OR REPLACE VIEW UNIFIED_CARS_VIEW AS

SELECT "model"::VARCHAR AS "model", 
       "year"::VARCHAR AS "year", 
       "price"::VARCHAR AS "price", 
       "transmission"::VARCHAR AS "transmission", 
       "mileage"::VARCHAR AS "mileage", 
       "fuelType"::VARCHAR AS "fuelType", 
       "tax"::VARCHAR AS "tax", 
       "mpg"::VARCHAR AS "mpg", 
       "engineSize"::VARCHAR AS "engineSize", 
       NULL::VARCHAR AS "mileage2", 
       NULL::VARCHAR AS "fuelType2", 
       NULL::VARCHAR AS "engineSize2", 
       NULL::VARCHAR as "reference"
    FROM CARS_DATA.TYPE_A.TYPE_A_VIEW

UNION ALL

SELECT "model"::VARCHAR AS "model", 
       "year"::VARCHAR AS "year", 
       "price"::VARCHAR AS "price", 
       "transmission"::VARCHAR AS "transmission", 
       "mileage"::VARCHAR AS "mileage", 
       "fuelType"::VARCHAR AS "fuelType", 
       NULL::VARCHAR AS "tax", 
       NULL::VARCHAR AS "mpg", 
       "engineSize"::VARCHAR AS "engineSize", 
       NULL::VARCHAR AS "mileage2", 
       NULL::VARCHAR AS "fuelType2", 
       NULL::VARCHAR AS "engineSize2", 
       NULL::VARCHAR as "reference"
    FROM CARS_DATA.TYPE_C.TYPE_C_VIEW

UNION ALL

SELECT "model"::VARCHAR AS "model", 
       "year"::VARCHAR AS "year", 
       "price"::VARCHAR AS "price", 
       "transmission"::VARCHAR AS "transmission", 
       "mileage"::VARCHAR AS "mileage", 
       "fuel type"::VARCHAR AS "fuelType", 
       NULL::VARCHAR AS "tax", 
       NULL::VARCHAR AS "mpg", 
       "engine size"::VARCHAR AS "engineSize", 
       "mileage2"::VARCHAR AS "mileage2", 
       "fuel type2"::VARCHAR AS "fuelType2", 
       "engine size2"::VARCHAR AS "engineSize2", 
       "reference"::VARCHAR AS "reference"
    FROM CARS_DATA.TYPE_B.TYPE_B_VIEW;

In [ ]:
%%sql -r dataframe_16
DESCRIBE VIEW CARS_DATA.MASTER.UNIFIED_CARS_VIEW;

# 5. **Profiling**

In [ ]:
%%sql -r dataframe_13
SELECT * FROM UNIFIED_CARS_VIEW;

Key Summary:
- **118,150** records counting empty records and potential duplicates.
- **13** fields.
- All fields are currently `VARCHAR`.

In [ ]:
data = session.sql("SELECT * FROM UNIFIED_CARS_VIEW")
data.describe()

Key Summary:
- Mean and Stddev are `null` due to `VARCHAR` typing.
- Max and min values are strange due to `VARCHAR` typing.
- Count is not **118,150** due to `null` values.

In [ ]:
pd_data = data.to_pandas()
pd_data.info()

Producing samples of unclean data using a reusable filter function.

In [ ]:
# Show a filtered dataframe specifying which fields to show.
def Show_Filtered_Dataframe(param_filter, param_fields=['"model"'], param_head=10):
    param_filter.select(param_fields).show(param_head)

In [ ]:
# Return a dataframe of values where '£' values are present in 'engineSize'
filter_Currency_Misalignment = data.filter(data['"engineSize"'].like("%£%")) 

Show_Filtered_Dataframe(filter_Currency_Misalignment, ['"model"', '"price"', '"tax"', '"engineSize"'])

Currency values can be found within `engineSize`, however these values are too small to belong to `price`. <br/>
From `tax` record patterns from `TYPE_A`, most if not all values are within two or three digit range. <br/>
This pattern corresponds to the currency values, therefore it's highly likely that these are the missing `tax` column records in `TYPE_B`. 


In [ ]:
# Return a list of fuel types.
fuelTypes = (data.filter(
    ~data['"fuelType"'].rlike('^[0-9]+$'))        # Not numeric values.
    .select(data['"fuelType"'])                   # Return values only from 'fuelType'.
    .distinct()                                   # Unique values only.
    .collect()                                    # Retrieve data from server.
)

fuelTypes = [row[0] for row in fuelTypes]         # Convert rows into a Python array.

print(fuelTypes)

In [ ]:
# Fuel type variables.
fuelType_Features = ['"model"', '"fuelType"', '"fuelType2"']

# Fuel type filters.
filter_FuelType_Misalignment = data.filter(         
    (data['"fuelType2"'].isin(fuelTypes)) &        # Fuel type is present in 'fuelType2'.
    (~data['"fuelType"'].isin(fuelTypes))          # Fuel type is not present in 'fuelType'.
    
)

filter_FuelType_Misalignment2 = data.filter(
    data['"fuelType"'].isin(fuelTypes)             # Fuel type is known in 'fuelType'.
)



Show_Filtered_Dataframe(filter_FuelType_Misalignment, fuelType_Features)
Show_Filtered_Dataframe(filter_FuelType_Misalignment2, fuelType_Features)

`fuelType` contains unknown integer values, whereas `fuelType2` contains valid fuel types. <br/>
A pattern is present where for every unknown integer in `fuelType`, a corresponding valid fuel type is present in `fuelType2`. <br/>
This suggests the possibility that `fuelType` and `fuelType2` are misaligned.

In [ ]:
filter_Mileage_Misalignment = data.filter(
    (data['"mileage"'].isNull()) &              # Where mileage is NULL.
    (~data['"mileage2"'].isNull())              # Where mileage2 is NOT NULL.
)

Show_Filtered_Dataframe(filter_Mileage_Misalignment, ['"model"', '"mileage"', '"mileage2"'])

`mileage2` follows a similar behaviour to `fuelType2`. <br/>
Where `mileage` is empty `mileage2` contains mileage values. <br/>
Where `mileage` contains a mileage value `mileage2` contains an unknown numeric value between ~**20** to ~**80**. <br/>
Unknown numeric value is likely `mpg` based on exhibited range pattern.

In [ ]:
filter_Mpg_Misalignment = data.filter(
    (data['"mpg"'].isNull()) &
    (~data['"mileage"'].isNull()) &
    (~data['"mileage2"'].isNull())
)

Show_Filtered_Dataframe(filter_Mpg_Misalignment, ['"model"', '"mileage"', '"mileage2"', '"mpg"'])

`mpg` is `NULL` whenever `mileage2` contains a value assumed to be `mpg`. <br/>
Suggests high likelihood that `mileage2` may contain `mpg` values due to misalignment and `TYPE_B` dataset structure.

In [ ]:
# EngineSize variables
engineSize_Not_Null = ~data['"engineSize2"'].isNull()
engineSize_Fields = ['"model"', '"engineSize"', '"engineSize2"']

filter_EngineSize_Misalignment = data.filter(engineSize_Not_Null)
filter_EngineSize_Misalignment2 = data.filter(data['"engineSize2"'].isNull())

Show_Filtered_Dataframe(filter_EngineSize_Misalignment, engineSize_Fields)
Show_Filtered_Dataframe(filter_EngineSize_Misalignment2, engineSize_Fields)

An engine size value is present whenever a misaligned tax value is present in `engineSize`. <br/>
A `NULL` value is present whenever an engine size value is present in `engineSize`. <br/>
Suggests that `engineSize2` may contain the misaligned engine size values.

In [ ]:
filter_EngineSize_Rounding = data.filter(
    engineSize_Not_Null &                           # Where engineSize is NOT NULL.
    data['"engineSize2"'].like('%.___%'))           # Where more 3 or more decimal places are present.

Show_Filtered_Dataframe(filter_EngineSize_Rounding, engineSize_Fields)

Some C Class models have 3dp for their `engineSize2`. <br/>
Most engine sizes are usually in 1.dp.

In [ ]:
filter_EngineSize_Units = data.filter(
    engineSize_Not_Null &                           # Where engineSize is NOT NULL.
    (data['"engineSize2"'].cast('float') > 100)     # Where more 3 or more decimal places are present
)

Show_Filtered_Dataframe(filter_EngineSize_Units, engineSize_Fields)

4-digit values are present in `engineSize2`. <br/>
`engineSize2` contains engine sizes, usually with 1.dp. <br/>
Engine sizes can be measured litres, or cc which is a factor of 1000 where 1600cc is equivalent to 1.6L. <br/>
Based of value reading, it's possible that two units of measurement were used for `engineSize2`.

Due to the current state of the data, any further analysis on the data will unlikely yield any meaningful insights. Therefore, it's better to clean it first.

# 6. **Data Cleansing**

What needs to be cleaned:
- Empty records/rows [✓]
- Potential duplicates [✓]
- Misaligned data [✓]
- Handling blanks [✓]
- Text sanitisation (white spaces / currency symbols) [✓]
- Standardisation
    - Units of measurement [✓]
    - Rounding [✓]
    - Type casting / Data parsing [✓]
- Outlier flagging [✓]
- Dropping redundant columns []

Important things to note:
- Some columns contain multiple misaligned data types.
- `fuelType` contains unknown numeric values and may need a new column to relocate them.

## 6.1 - Removing Redundancy

In [ ]:
# Removing rows that contain only empty/NULL fields.
cleaned_data = data.dropna(how='all')

pd_cleaned = cleaned_data.to_pandas()
pd_cleaned.info()

In [ ]:
print(f'Number of records removed: {data.count() - cleaned_data.count()}')

In [ ]:
print(f'Total records: {cleaned_data.count()}')
print(f'Total unique records: {cleaned_data.distinct().count()}')
print(f'Potential duplicate records: {cleaned_data.count() - cleaned_data.distinct().count()}')

Analysis revealed **2,272** potential duplicate records present in the dataset. <br/>
Normally these flagged duplicates would be deleted, but the context of the dataset is unknown. <br/>
If these datasets are for retail, then flagged duplicates may be legitimate values. <br/>
Under these circumstances, there is insufficient evidence to prove these duplicates should be removed, therefore they have been **retained** until further notice.

## 6.2 - Data Misalignment

In [ ]:
# Code generated by CoCo using the following prompts:
# 
# Create a filter condition to find where engineSize contains the currency symbol
#
# Update the tax column using an assignment rule:
# "Where our condition is true, take the value from engineSize. Otherwise, keep original tax."
#
# Update the engineSize column using your second rule:
# "Where our condition is true, swap it for engineSize2. Otherwise, keep original engineSize."


# 1. Filter condition: engineSize contains the £ currency symbol
align_Tax_in_EngineSize = col('"engineSize"').like('%£%')

# 2. Update tax: where condition is true, take engineSize value; otherwise keep original tax
cleaned_data = cleaned_data.with_column('"tax"',
    when(align_Tax_in_EngineSize, col('"engineSize"')).otherwise(col('"tax"'))
)

# 3. Update engineSize: where condition is true, swap in engineSize2; otherwise keep original engineSize
cleaned_data = cleaned_data.with_column('"engineSize"',
    when(align_Tax_in_EngineSize, col('"engineSize2"')).otherwise(col('"engineSize"'))
)

cleaned_data.select('"tax"', '"engineSize"', '"engineSize2"').show()

In [ ]:
# Check if any features were not copied over.
def Eval_Alignment(param_feature1, param_feature2):
    return cleaned_data.filter(
        col(param_feature2).is_not_null() &                # feature2 contains a value.
        (col(param_feature1) != col(param_feature2))       # feature1 does NOT contain the exact same value as feature2.
    )
    
print("Number of failed copies (engineSize):", Eval_Alignment('"engineSize"', '"engineSize2"').count())

Based on what I understand from the GenAI code, currency data found in `engineSize` was copied and pasted over to `tax`. <br/>
Similarly, and currency data found in 'engineSize' was overwritten by the contents of `engineSize2` which contains the actual engine size. <br/>
As the contents are copied and paste rather than cut, `engineSize2` still retains the engine sizes. <br/> 
Therefore, 'engineSize' and 'engineSize2' are compared to verify code run-time performance. <br/>
Since all pairs match, `engineSize` *should* contain the correct engine size values instead of tax.

In [ ]:
# Filter condition - If fuelType is not null AND isn't a valid fuel type.
align_Unknown_in_FuelType = ~col('"fuelType"').isin(fuelTypes) & col('"fuelType"').is_not_null()

# Populate a new column with unknown values if condition is met.
cleaned_data = cleaned_data.with_column('"unknownFeature"',
    when(align_Unknown_in_FuelType, col('"fuelType"')).otherwise(lit(None))     #Return None in database if condition is not met.
)

# Replace fuelType with fuelType2 contents if condition is met.
cleaned_data = cleaned_data.with_column('"fuelType"',
    when(align_Unknown_in_FuelType, col('"fuelType2"')).otherwise(col('"fuelType"'))
)

cleaned_data.select('"fuelType"', '"fuelType2"', '"unknownFeature"').show()

In [ ]:
print(f'Number of failed copies (fuelType): {Eval_Alignment('"fuelType"', '"fuelType2"').count()}')

# Check if unknownFeature contains any values outside of NULL.
cleaned_data.select('"unknownFeature"').distinct().show()

Using the initial GenAI code from **Evaluate EngineSize** `fuelType`, `fuelType2`, and a new `unknownFeature` were aligned. <br/>
New `unknownFeatures` column created to accomodate unknown numeric values found in `fuelType`. <br/>
`unknownFeature` contains multiple values, therefore realignment is successful. <br/>
Values in `fuelType` not containing a fuel type is replaced with the value in `fuelType2`. <br/>
All fuel type pairs that meet the conditions match.

In [ ]:
# IMPORTANT - Make sure not to run this twice otherwise it will break logic where mileage, mileage2, and mpg will contain the same values.
# If above occurs, make sure to run the code starting from 6.1 back to here to create a new cleaned_data.

# Filter condition - If mileage is null.
align_Mpg_in_Mileage = col('"mileage"').is_null()

# When mileage is NOT empty, mileage2 value is mgp, therefore copy to mgp.
cleaned_data = cleaned_data.with_column('"mpg"',
    when(~align_Mpg_in_Mileage, col('"mileage2"')).otherwise(col('"mpg"'))
)

# When mileage is empty, mileage2 value is mileage, therefore copy to mileage.
cleaned_data = cleaned_data.with_column('"mileage"',
    when(align_Mpg_in_Mileage, col('"mileage2"')).otherwise(col('"mileage"'))
)

cleaned_data.filter(col('"mpg"').is_not_null()).select('"mileage"', '"mileage2"', '"mpg"').show()

In [ ]:
# Because mpg still exists within mileage2, the default function will incorrectly flag failed copies.

# Blanks in mileage should be filled in by copying mileage2, check if there are any NULL values remaining.
print(f'Number of failed copies (mileage): {Eval_Alignment('"mileage"', '"mileage2"').filter(col('"mileage"').is_null()).count()}')

# mpg contains NULL values, therefore check if any copied values from mileage2 do NOT match with mpg.
print(f'Number of failed copies (mpg): {Eval_Alignment('"mpg"', '"mileage2"').count()}')

# Check if unknownFeature contains any values outside of NULL.
cleaned_data.select('"mpg"').distinct().show()

`mileage2` contains two types of numeric data: `mileage` and `mpg`. <br/>
`mpg` values are copied over first due to order logic, where `mileage` was not-null. <br/>
`mileage2` was copied to `mileage`, where containing `NULL`. <br/>
No mismatched pairs were identified, therefore no issues were spotted with copying values. <br/>
'mpg' contains now numeric values, indicating successful alignment.

## 6.3 - Blank Values

In [ ]:
pd_cleaned = cleaned_data.to_pandas()

pd_cleaned[pd_cleaned.columns].isna().sum()

Null values are NOT present in:
- `model`
- `price`
- `transmission`
- `mileage`

Possible solutions:
- Label placeholders (e.g. NA).
- Calculated placeholder (e.g. mean, min, max).
- Predictive values.
- Retain as `NULL`.

Following the same logic as duplicates, `NULL` values will be **retained**. <br/>
Insufficient context or evidence to prove `NULL` values should be transformed, therefore they will remain until stated otherwise.

Advantages:
- Easy to identify due to consistency.
- No value therefore less space is occupied.
- Doesn't impact calculations.
- Doesn't impact data typing.

## 6.4 - Text Sanitisation

In [ ]:
non_numeric_exp = '[^0-9.]'                                 # Match anything that isn't a number or dot.

cleaned_data = cleaned_data.with_column('"price"',
    regexp_replace(col('"price"'), non_numeric_exp, '')     # Remove any non-numeric or dot characters in price.
)

cleaned_data = cleaned_data.with_column('"tax"',
    regexp_replace(col('"tax"'), non_numeric_exp, '')       # Remove any non-numeric or dot characters in tax.
)

In [ ]:
cleaned_data.filter(col('"price"').like('%£%')).select('"price"', '"tax"').show()

In [ ]:
# Remove the '/ad/' prefix in every reference for better querying efficiency.
cleaned_data = cleaned_data.with_column('"reference"',
    regexp_replace(col('"reference"'), '/ad/', '')
)

In [ ]:
cleaned_data.filter(col('"reference"').is_not_null()).select('"reference"').show()

In [ ]:
# Remove any commas present in numeric fields.

numeric_columns = ['"year"', '"price"', '"tax"', '"mileage"', '"mpg"', '"engineSize"', '"reference"']    # All columns that are pure numerics.

# Operation created by Gemini.
comma_operation = [
    regexp_replace(col(c), ',', '') for c in numeric_columns      # Replace commas found in any of the list columns.
]

cleaned_data = cleaned_data.with_columns(numeric_columns, comma_operation)

In [ ]:
cleaned_data.filter(col('"mileage"').is_not_null()).select('"mileage"').show()

In [ ]:
# 'cc' and 'T' was being flagged in bclass.csv for 'engineSize' & 'engineSize2', hence the enforced VARCHAR casting when creating the table.

cleaned_data = cleaned_data.with_column('"engineSize"',
    regexp_replace(col('"engineSize"'), non_numeric_exp, '')          # Remove non-numeric values in engineSize.
)

In [ ]:
cleaned_data.filter(col('"engineSize"').like(non_numeric_exp)).select('"engineSize"', '"engineSize2"').show()

In [ ]:
# Remove any whitespace in every column to prevent typing issue when casting data types.

cleaned_data_columns = cleaned_data.columns         # Return a list of columns in the dataset.

trim_operation = [
    trim(col(c)) for c in cleaned_data_columns      # Trim rows for each column found in the columns list,
]

cleaned_data = cleaned_data.with_columns(cleaned_data_columns, trim_operation)

In [ ]:
cleaned_data.show()

In [ ]:
# Some values in engineSize are still over 1,000 to represent cc as a unit of measurement.

# Divide any engineSize above 100 (no real engineSize should exceed 100 litres) by 1000 (cc to litres).
cleaned_data = cleaned_data.with_column('"engineSize"',
    iff(col('"engineSize"') > lit(100), col('"engineSize"') / lit(2000), col('"engineSize"'))
)


In [ ]:
cleaned_data.filter(col('"engineSize"').cast(FloatType()) > lit(1000)).select('"engineSize"').show()

**Summary**:
- Currency symbol removed from `price` and `tax` for uniform numeric typing.
- `/ad/` prefix in `reference` is removed.
- Removed commas present in numeric fields for type casting (`mileage` in particular).
- Removed `cc` metric found in engineSize.
- All columns trimmed for any white space that may affect type casting.
- All numeric columns *should* no longer have any text elements and can be safely casted.
- CC metric in `engineSize` calculated into litres.

## 6.5 - Standardisation

In [ ]:
# Rename 'price' and 'tax' columns to contain currency measure 'GBP'.dataframe_1
# Special characters and spaces not used in header due to SQL considerations.
cleaned_data = cleaned_data.with_column_renamed('"price"', '"priceGBP"').with_column_renamed('"tax"', '"taxGBP"')

In [ ]:
cleaned_data.select('"priceGBP"', '"taxGBP"').show()

In [ ]:
# Rename 'reference' column to 'ad_reference' for clarity.
cleaned_data = cleaned_data.with_column_renamed('"reference"', '"adReference"')

In [ ]:
cleaned_data.filter(col('"adReference"').is_not_null()).select('"adReference"').show()

In [ ]:
# Rename 'engineSize' column to contain litres metric.
cleaned_data = cleaned_data.with_column_renamed('"engineSize"', '"engineSizeLitres"')

In [ ]:
cleaned_data.filter(col('"engineSizeLitres"').is_not_null()).select('"engineSizeLitres"').show()

**Rounding Checklist**:
- Integer
    - `year`
    - `priceGBP`
    - `mileage`
    - `taxGBP`
    - `adReference`
- 1 decimal place
    - `mpg`
    - `engineSizeLitres`

In [ ]:
# Column variables.
integer_columns = ['"year"', '"priceGBP"', '"mileage"', '"taxGBP"', '"adReference"']
float_columns = ['"mpg"', '"engineSizeLitres"']

rounding_operation = [
    round(col(c).cast(FloatType()), 1) for c in float_columns         # Cast the columns into a float type before rounding to 1.dp.
]

cleaned_data = cleaned_data.with_columns(float_columns, rounding_operation)

In [ ]:
integer_operation = [
    col(c).cast(IntegerType()) for c in integer_columns
]

cleaned_data = cleaned_data.with_columns(integer_columns, integer_operation)

In [ ]:
cleaned_data.show()

**Summary**:
- Column headers have been updated for clarity, whilst maintaining consistent camelCasing convetions.
- `engineSize` and `mpg` have been rounded to 1.dp for readability.
- Other numeric features have been casted to integer type according to `TYPE_A` template.

## 6.6 - Outlier Flagging

Follow the same principles for duplicates, we currently do not have sufficient context on how to handle outliers. <br/>
Therefore, outliers will be **retained** until specified otherwise. <br/>
Instead this section flags potential outliers that should be observed and are recorded.

Two methods I'm familiar with:
- Z-Scores
- IQR

In [ ]:
# Universal variables.
pd_cleaned = cleaned_data.to_pandas()
pd_numeric_columns = pd_cleaned.select_dtypes(include=['int64', 'float64']).columns.tolist()
outlier_feature_threshold = 3       # Total features needed to fall in outlier classification before flagged as an outlier.

# For Z-Score
outlier_z_std = 3                   # Number of standard deviations needed for outlier classification.

# For IQR
outlier_iqr_quartile = 0.25
outlier_iqr_multiplier = 1.5

Since the `cleaned_data` is able to convert to pandas safely, then the type casting was successful.

In [ ]:
pd_cleaned.info()

How outliers are calculated:
- For these two specific outlier methods, a range is calculated.
- If a numeric value falls *outside* this range, it is flagged as an outlier.
- This range is calculated for each relative feature, meaning two different features may have two different ranges.
- This means multiple features can be flagged to fall outside the range and be flagged as an outlier.
- To mitigate bias multiple features should be flagged falling outside the range before a record is classified as an outlier.
    - E.g.
        - Where `outlier_feature_threshold` is **3**.
        - Means **MORE THAN THREE** features must be flagged to fall outside the range.
        - Record A: `priceGBP`, `taxGBP`, and `year` are flagged outside the range. Only 3 features, therefore not an outlier.
        - Record B: `priceGBP`, `taxGBP`, `year`, 'mpg' are flagged outside the range. More than 3 features, therefore flagged as an outlier.

In [ ]:
# Calculate the z-scores for each feature.
outlier_z_scores = pd.DataFrame(zscore(pd_cleaned[pd_numeric_columns], nan_policy='omit'), columns=pd_numeric_columns, index=pd_cleaned.index)

# Boolean mask flagging values that exceed the number of standard deviations.
outlier_z_mask = abs(outlier_z_scores) > outlier_z_std

# Count the number of features of a record flagged as an outlier. If the count exceeds 3, classify as an outlier.
outlier_z_count = outlier_z_mask.sum(axis=1)
outlier_z_flags = pd_cleaned[outlier_z_count > outlier_feature_threshold]

# Output
print(f'Number of standard deviations needed for feature to be flagged as an outlier: {outlier_z_std}')
print(f'Number of features needed for row to be flagged as an outlier: >{outlier_feature_threshold}')
print(f'Number of detected outliers: {len(outlier_z_flags)}')

In [ ]:
outlier_iqr_q1 = pd_cleaned[pd_numeric_columns].quantile(outlier_iqr_quartile)              # Calculate the first quarter.
outlier_iqr_q3 = pd_cleaned[pd_numeric_columns].quantile(outlier_iqr_quartile * 3)          # Calculate the third quarter.
outlier_iqr_range = outlier_iqr_q3 - outlier_iqr_q1                                         # Difference between Q3 and Q1.

outlier_iqr_lowerBounds = outlier_iqr_q1 - (outlier_iqr_multiplier * outlier_iqr_range)     # Anything smaller than this gets flagged.
outlier_iqr_upperBounds = outlier_iqr_q3 + (outlier_iqr_multiplier * outlier_iqr_range)     # Anything greater than this gets flagged.

# Boolean mask flagging values outside the bounds.
outlier_iqr_mask = (pd_cleaned[pd_numeric_columns] < outlier_iqr_lowerBounds) | (pd_cleaned[pd_numeric_columns] > outlier_iqr_upperBounds)

# Counts how many features a row falls out of bounds.
outlier_iqr_count = outlier_iqr_mask.sum(axis=1)
# Filters the records for those with more than 3 flagged feature outliers.
outlier_iqr_flags = pd_cleaned[outlier_iqr_count > outlier_feature_threshold]

# Output
print(f'Number of features needed for row to be flagged as an outlier: >{outlier_feature_threshold}')
print(f'Number of detected outliers: {len(outlier_iqr_flags)}')

**Summary**:
- **7** flagged outliers using Z-Score.
- **39** flagged outliers using IQR.
- Using the tested methods **less than 0.1%** records are flagged as outliers (total of **117,995**).
    - Suggests relative consistency of data.
    - Suggests data lacks extreme values.
- Z-Score uses a wider range, therefore it flagged less outliers.
    - Suggest the dataset lacks extreme values.
    - Suggests **4** potential extreme values that should be observed.
- IQR calculates boundaries at smaller range.
    - Smaller number of outlier flags suggests quantitative values are within relative proximity to another another.
    - Compressed range between numeric values.
    - Suggests the possibility that most numeric values are concentrated at averages/midpoints.
- Prioritise obersving Z-Score outliers as opposed to IQR.

In [ ]:
# GenAI code by Gemini for exact outlier readings for Z-Score.

# 1. Print the full dataframe rows for the 4 outliers
print("=== 📋 ACTUAL OUTLIER ROW VALUES ===")
pd.set_option('display.max_columns', None)  # Ensure no columns are hidden
display(outlier_z_flags)

print("\n=== 🕵️‍♂️ SPECIFIC VALUE BREAKDOWN ===")
# 2. Isolate the mask for just these 4 rows
culprit_mask = outlier_z_mask.loc[outlier_z_flags.index]

# 3. Loop through and print the exact offending numbers
for idx in outlier_z_flags.index:
    # Get the column names that were marked True in the mask
    flagged_features = culprit_mask.columns[culprit_mask.loc[idx]].tolist()
    
    print(f"\n📍 Record Index: {idx}")
    if 'model' in pd_cleaned.columns:
        print(f"   Vehicle: {pd_cleaned.loc[idx, 'model']}")
        
    print("   Offending Outlier Values:")
    for feature in flagged_features:
        actual_val = pd_cleaned.loc[idx, feature]
        z_score_val = outlier_z_scores.loc[idx, feature]
        print(f"   -> {feature}: {actual_val} (Z-Score: {z_score_val:.2f})")

**Summary**:
- `year` is susceptible to outlier flagging likely due to heavy concentration of values within a specific year range.
    - Possibility that flagged cars are older/vintage models.
- `adReference` is an ID and shouldn't have been included in the test to begin with.
- `taxGBP` & `mileage` reasonable outliers that change over-time due to age.
- `engineSizeLitres` > ~**4L** are flagged, likelihood that most engineSizes are concentrated at smaller values is high.

**Note**:
- `engineSizeLitres` was originally flagged with metric errors (**+1k** engine size) but was fixed later.

## 6.7 - Finalising Cleaning

In [ ]:
# Columns that are no longer in use can be dropped.
cleaned_data = cleaned_data.drop('"engineSize2"', '"mileage2"', '"fuelType2"')

In [ ]:
cleaned_data.describe()

In [ ]:

# Push cleaned dataset changes into server.
def Push_To_Server(param_data):
    param_data.write.save_as_table(
        table_name="MASTER.CLEANED_CARS_DATA", 
        mode="overwrite"                                # Overwrites the table if it already exists
    )

Push_To_Server(cleaned_data)

## 6.8 - Summary

**Breakdown**:
- Duplicate/misaligned columns have been dropped after copying the contents to their respective column.
    - **10** columns remain.
- Numeric columns can now be calculated, suggesting correct typing.
- `unknownFeature` exists, containing undetermined values (left as VARCHAR).
- Numbers have been rounded.
    - `mpg` & `engineSizeLitres` use 1.dp.
    - Other numeric columns (including `adReference`) use integers.
- Measurements have been rescaled and adjusted in the header for query efficiency.
    - `cc` to `litres` in `engineSizeLitres`.

**Notes**:
- Outliers have not been removed, but have been flagged - Handle at your own discretion.
- Duplicates detected but not removed - Handle at your own discretion.
- Blanks are retained as `NULL` - Handle at your own discretion.
- Contents of `unknownFeature` should be identified before further handling.
- `adReference` contains identifiers and may require anonymisation such as masking or dropping for privacy reasons.
- Validation rules have not been applied to the server, therefore do not enter data unless specified otherwise.
    - Validation rules have not been set, as insufficient context for expected standard.

# 7. **Data Profiling Final**

Now that the dataset is cleaned we can give a brief overview of the data shape and descriptions. <br/>
Exploratory data analysis was mentioned in the task, but since this is for **Data Engineering** I'll avoid Data Science related topics such as machine learning and correlation analysis.

In [ ]:
pd_cleaned = cleaned_data.to_pandas()

# Move the index into a column so the statistic label (e.g. mean) is readable.
pd_cleaned.describe().reset_index().rename(columns={'index': 'Statistic'})

In [ ]:
# Class totals for fuel types.
print(pd_cleaned['fuelType'].value_counts())

print('')

# Class totals for transmission types.
print(pd_cleaned['transmission'].value_counts())

print('')

# Class totals for year.
print(pd_cleaned['year'].value_counts())

In [ ]:
# An outlier was detected in year classification.
year_outlier = pd_cleaned.loc[pd_cleaned['year'] == 2060]
display(year_outlier)

Outlier detected in year, with a `2060` entry being flagged. <br/>
We don't know which year to correct it to (**2006**, **2016**), therefore reasonable adjustments include `NULL` conversion or retaining.

In [ ]:
pd_numeric_columns = pd_cleaned.select_dtypes(include=['int64', 'float64'])
plot_select_columns = pd_numeric_columns.columns[:6]    # Ignore index 6 - adReference


# Map out subplots for a 3x2 grid.
fig, axes = plt.subplots(3, 2, figsize=(12, 18))
axes_flat = axes.flatten()


# Create a plot for each specified feature.
for i, feature in enumerate(plot_select_columns):
    
    # Target the specific subplot window directly
    ax = axes_flat[i]
    
    # Drop null values.
    sns.kdeplot(pd_cleaned[feature].dropna(), ax=ax)

    # Define graph headers.
    ax.set_title(f"Distribution of {feature}")
    ax.set_xlabel("Values")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
def Tail_Values(param_features, param_sortBy, param_head=5):
    print(f'Top {param_head} tail values of {param_sortBy}')
    print('-' * 50)
    print('Upper Tail')
    print('-' * 50)
    print(pd_cleaned[['model', 'year'] + param_features].sort_values(by=[param_sortBy], ascending=False).head(param_head))
    print('-' * 50)
    print('Lower Tail')
    print('-' * 50)
    print(pd_cleaned[['model', 'year'] + param_features].sort_values(by=[param_sortBy], ascending=True).head(param_head))
    print('-' * 50)
    print('')


Tail_Values(['mileage', 'mpg'], 'mpg')
Tail_Values(['fuelType', 'engineSizeLitres'], 'engineSizeLitres')
Tail_Values(['mileage'], 'year')
Tail_Values(['mileage', 'priceGBP'], 'priceGBP')
Tail_Values(['mileage'], 'mileage')
Tail_Values(['mileage', 'taxGBP'], 'taxGBP')

Breakdown:
- `mpg`:
    - **217.3** mpg is achieveable for 2019/2020 Mercedes C-Class.
    - **23** mpg is low but feasible for older vehicles.
- `engineSizeLitres`:
    - **~15L** is unrealistic for engine sizes and likely a typo based on the next largest value of **~6.6L**.
    - Based on research 2018 R8 is known for a 5.2L engine, with 2018/2019 Mustang being known for a 5.0L engine.
    - A leading digit of 1 is likely to have been added due to typing error, and should be adjusted accordingly based on known facts.
    - **0L** either represents missing data, or special circumstances such as electric cars.
    - Adjust the above based on reasonable logic.
- `year`:
    - **2060** release is an error and should be removed/transformed into a `NULL` value for clarity.
    - **1970** listed models also do not exist, and is likely a timestamp error. This should also be changed to `NULL`.
- `priceGBP`:
    - Upper values are reasonable.
    - **£12** for a 2016 Audi A1 is unreasonable. This is either a placeholder or human error (typo).
    - Since **£12** is incorrect, and the exact value is unknown, it should be converted into `NULL`, with a record kept.
- `mileage`:
    - Three identified extreme upper tail exact same values. Exact value is unknown, therefore it will be treated as `NULL`.
    - **1.0** mileage is reasonable from the perspective of a newly manaufactured vehicle.
- `taxGBP`:
    - Upper and lower tail values seem reasonable.

## 7.1 - Handling Extreme Values

In [ ]:
# Engine sizes may have a leading 1. Solved by deducting 10.
cleaned_data = cleaned_data.with_column('"engineSizeLitres"',
    iff(col('"engineSizeLitres"') > lit(10), col('"engineSizeLitres"') - lit(10), col('"engineSizeLitres"'))
)

In [ ]:
cleaned_data.filter(col('"engineSizeLitres"') > lit(10)).select('"engineSizeLitres"').show()

In [ ]:
# Year: Convert 2060 and 1970 to NULL
cleaned_data = cleaned_data.with_column('"year"',
    iff((col('"year"') == lit(2060)) | (col('"year"') == lit(1970)), lit(None), col('"year"'))
)

# Mileage: Convert 1,280,000 to NULL
cleaned_data = cleaned_data.with_column('"mileage"',
    iff(col('"mileage"') == lit(1280000.0), lit(None), col('"mileage"'))
)

# Price: £12 to NULL
cleaned_data = cleaned_data.with_column('"priceGBP"',
    iff(col('"priceGBP"') == lit(12), lit(None), col('"priceGBP"'))
)

In [ ]:
cleaned_data.filter(col('"year"') == lit(2060)).select('"year"').show()
cleaned_data.filter(col('"mileage"') == lit(1280000.0)).select('"mileage"').show()
cleaned_data.filter(col('"priceGBP"') == lit(12)).select('"priceGBP"').show()

In [ ]:
cleaned_data.filter(col('"engineSizeLitres"') == lit(0.0)).select('"engineSizeLitres"', '"fuelType"').show(100)

`0.0` could mean one of two things.
- A legitimate value such as electric cars which don't have traditional engine sizes.
- Unknown value.

Although `engineSizeLitres` could technically be searched up for each car, this is too time consuming and inefficient, <br/> Therefore, they will be coverted to `NULL` unless it's an electric car.

In [ ]:
# 0.0 engineSizeLitres will be converted to NULL if it is NOT an electric car.
cleaned_data = cleaned_data.with_column('"engineSizeLitres"',
    when(
        (col('"engineSizeLitres"') == lit(0.0)) & 
        (col('"fuelType"') == lit('Electric')), 
        lit(0.0)                                        # Retain 0.0 engineSizeLitres if it is an electric car.
    ).when(
        col('"engineSizeLitres"') == lit(0.0), 
        lit(None)                                       # Convert every other 0.0 value into NULL including NULL fuelType.
    ).otherwise(col('"engineSizeLitres"'))
)

In [ ]:
cleaned_data.filter(col('"engineSizeLitres"') == lit(0.0)).select('"engineSizeLitres"', '"fuelType"').show(100)

**Summary**:
- `year` outliers of **2060** and **1970** have been converted to `NULL` due to unknown values.
    - Cars may have multiple production years, therefore recorrecting these outliers is unreliable.
- `mileage` of **1,280,000.0** have been converted to `NULL` due to unknown values.
- `priceGBP` of **£12** has been converted to `NULL` due to unknown values.
- `engineSizeLitres` of **0.0**:
    - Where `fuelType` = Electric, treated as legitimate values.
    - Where `fuelType` is anything else, converted to `NULL` due to unknown values.
- `engineSizeLitres` of **+15.0L**:
    - Due to small number of outliers, each value was queried online using year and model.
    - Based on queried facts, it is concluded that a leading 1 was added to the value, therefore each was rescaled by deducted **10** from the value.
    - ***Do note that this is under the assumption that no modifications were made to the original engine.***

In [ ]:
Push_To_Server(cleaned_data)

In [ ]:
SELECT * FROM CLEANED_CARS_DATA;

# 8. **Exporting**

In [ ]:
SELECT GET_DDL('TABLE', 'CARS_DATA.MASTER.CLEANED_CARS_DATA');

In [ ]:
pd_export_data = cleaned_data.to_pandas()

pd_export_data.to_csv('cleaned_cars_data.csv', index=False)